In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_301_Anand_Vihar_Delhi_DPCC_1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,199.56,335.12,38.99,56.29,61.65,18.28,9.33,1.60,17.08,...,NaN,11.83,75.30,0.41,60.27,NaN,0.0,127.21,741.94,NaN
1,2024-01-02,195.91,320.67,46.46,62.58,71.07,19.42,22.19,1.77,17.98,...,NaN,11.54,72.11,0.54,54.18,NaN,0.0,135.27,741.98,NaN
2,2024-01-03,247.44,360.37,69.63,71.08,94.41,25.29,16.65,2.18,15.52,...,NaN,10.76,84.56,0.51,38.12,NaN,0.0,119.43,742.00,NaN
3,2024-01-04,274.06,436.22,91.01,71.07,111.79,34.86,10.11,1.86,14.38,...,NaN,11.54,81.26,0.32,96.44,NaN,0.0,111.25,741.96,NaN
4,2024-01-05,218.28,345.98,66.03,59.13,85.10,42.52,11.37,2.20,15.25,...,NaN,12.01,84.88,0.32,51.01,NaN,0.0,79.26,742.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,201.56,240.34,56.41,93.86,95.78,98.62,6.33,1.29,7.82,...,NaN,13.85,85.92,1.60,240.99,NaN,0.0,18.47,740.51,NaN
362,2024-12-28,111.92,151.11,93.26,97.74,127.79,82.53,11.67,1.37,2.61,...,NaN,13.98,90.09,0.60,212.98,NaN,0.0,31.43,740.03,NaN
363,2024-12-29,123.29,178.12,57.66,75.84,87.24,72.08,1.54,1.23,3.85,...,NaN,13.84,84.74,1.05,222.97,NaN,0.0,41.78,740.00,NaN
364,2024-12-30,106.01,181.87,49.98,67.24,76.41,57.98,4.70,1.64,4.57,...,NaN,12.87,79.71,1.03,194.38,NaN,0.0,55.83,739.99,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 20)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 22
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (344, 19)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         199.56        335.12       38.99        56.29   
1  2024-01-02         195.91        320.67       46.46        62.58   
2  2024-01-03         247.44        360.37       69.63        71.08   
3  2024-01-04         274.06        436.22       91.01        71.07   
4  2024-01-05         218.28        345.98       66.03        59.13   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      61.65        18.28         9.33        1.60          17.08   
1      71.07        19.42        22.19        1.77          17.98   
2      94.41        25.29        16.65        2.18          15.52   
3     111.79        34.86        10.11        1.86          14.38   
4      85.10        42.52        11.37        2.20          15.25   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             2.31             8.15    11.83   75.30      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,0.934198,0.146262,-0.459685,-0.563365,-0.592890,-0.978328,-0.945065,-0.403237,-0.853990,0.220906,-0.919220,-1.805408,1.034930,-0.716229,-0.768530,0.0,-0.353910,0.635058
1,2024-01-02,0.888798,0.053431,-0.297657,-0.361157,-0.407468,-0.908128,0.738721,-0.259955,-0.823531,0.159293,-0.886134,-1.839296,0.814336,-0.182311,-0.843990,0.0,0.195774,0.645442
2,2024-01-03,1.529757,0.308475,0.204914,-0.087904,0.051953,-0.546659,0.013358,0.085606,-0.906786,0.848243,-0.771664,-1.930443,1.675276,-0.305523,-1.042984,0.0,-0.884499,0.650634
3,2024-01-04,1.860872,0.795756,0.668658,-0.088225,0.394059,0.042652,-0.842938,-0.184100,-0.945368,0.702611,-0.536639,-1.839296,1.447075,-1.085865,-0.320358,0.0,-1.442367,0.640250
4,2024-01-05,1.167048,0.216030,0.126828,-0.472066,-0.131303,0.514348,-0.677963,0.102463,-0.915924,0.467360,-0.693703,-1.784374,1.697404,-1.085865,-0.883268,0.0,-0.061678,0.650634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
339,2024-12-27,0.959075,-0.462631,-0.081836,0.644416,0.078920,-0.144856,-1.337861,-0.664515,-1.167381,1.654819,1.287654,-1.569362,1.769322,-0.059099,1.470718,0.0,-0.061678,0.263826
340,2024-12-28,-0.155918,-1.035870,0.717462,0.769148,0.709001,2.978124,-0.638684,-0.597089,-1.343706,1.324347,1.065179,-1.554171,2.057685,0.064113,1.123654,0.0,-0.061678,0.139216
341,2024-12-29,-0.014491,-0.862350,-0.054722,0.065118,-0.089180,2.334624,-1.965025,-0.715085,-1.301740,0.495366,0.608821,-1.570530,1.687723,1.912292,1.247437,0.0,-0.061678,0.131428
342,2024-12-30,-0.229430,-0.838259,-0.221306,-0.211350,-0.302356,1.466359,-1.551280,-0.369524,-1.277373,0.131287,0.350598,-1.683879,1.339890,1.830151,0.893187,0.0,-0.061678,0.128832


In [10]:
df.to_excel('Anandvihar2024.xlsx', index=False)